In [15]:
from ingest import load_faq_data
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI()

In [16]:
documents_all = load_faq_data()

[{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 471}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 253}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 118}]


In [17]:
documents = [doc for doc in documents_all if doc['course'] == 'llm-zoomcamp']

## structured output
forces an llm to output in a specific way

In [18]:
from pydantic import BaseModel

class Questions(BaseModel):
  questions: list[str]

In [19]:
data_gen_instructions = """
Emulate student who's taking course. Formulate 5 questions student might ask based on a FAQ record. Record should contain answer to the questions, and questions should be complete and not too short.
Use as few words as possible from record.

Output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [20]:
import json

doc = documents[0]
user_prompt = json.dumps(doc)

messages = [
  {"role": "system", "content": data_gen_instructions},
  {"role": "user", "content": user_prompt}
]

In [21]:
response = openai_client.responses.parse(
  model="gpt-5.4-mini",
  input=messages,
  text_format=Questions
)

In [22]:
response.output_parsed.questions

['Can I still join the course if I just found it now?',
 'Is it too late to start learning after the course has already begun?',
 'If I join late, can I still get the course certificate?',
 'What do I need to do to be eligible for a certificate if I’m starting now?',
 'Are late submissions for the final project accepted for the certificate?']

In [23]:
from evaluation_utils import llm_structured

In [28]:
def format_questions(questions, doc):
  records = []
  for q in questions:
    records.append({
      'question': q,
      'document': doc['id']
    })
  return records

In [29]:
import pandas as pd
from evaluation_utils import llm_structured_retry

In [30]:
def generate_ground_truth(doc):
  prompt = json.dumps(doc)
  result, usage = llm_structured_retry(openai_client, data_gen_instructions, prompt, Questions)
  return format_questions(result.questions, doc), usage



In [31]:
from tqdm.auto import tqdm

ground_truth = []
usages = []
for doc in tqdm(documents[:1]):
  ground_truth, usage = generate_ground_truth(doc)
  ground_truth.append(ground_truth)
  usages.append(usage)

  0%|          | 0/1 [00:00<?, ?it/s]

In [32]:
from evaluation_utils import map_progress
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
  results = map_progress(pool, documents, generate_ground_truth)



  0%|          | 0/118 [00:00<?, ?it/s]

In [43]:
ground_truth = []
usages = []

for records, usage in results:
  ground_truth.extend(records)
  usages.append(usage)


In [44]:
from evaluation_utils import calc_total_price

money_spent = calc_total_price(usages)

In [45]:
money_spent

0.09095250000000002

In [46]:
ground_truth[0]

{'question': 'I just found this course late — can I still sign up and follow along, or is it too late?',
 'document': '74eb249bbf'}

In [49]:
df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth.to_csv('data/ground_truth.csv', index=False)

In [50]:
df_ground_truth.head()

,question,document
0,I just found this course late — can I still si...,74eb249bbf
1,"If I join after the course already started, ca...",74eb249bbf
2,Do I have to submit the final project before s...,74eb249bbf
3,Is it okay to start the course now if I missed...,74eb249bbf
4,What’s the deadline for project submission if ...,74eb249bbf
